# Task1 - Environment Setup & Data Loading

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

DATA_DIR_CANDIDATES = [
    Path("data"),
    Path("../data"),
]

DATA_DIR = next(
    (path for path in DATA_DIR_CANDIDATES if (path / "drivers.csv").exists()),
    None,
)

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find data/drivers.csv. "
        "Run this notebook from the project root or from the jaeeun folder."
    )

OUTPUT_DIR = Path("jaeeun/output") if DATA_DIR == Path("data") else Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MISSING_VALUE = r"\N"

drivers = pd.read_csv(DATA_DIR / "drivers.csv")
races = pd.read_csv(DATA_DIR / "races.csv")
results = pd.read_csv(DATA_DIR / "results.csv")
constructors = pd.read_csv(DATA_DIR / "constructors.csv")
circuits = pd.read_csv(DATA_DIR / "circuits.csv")

dataframes = {
    "drivers": drivers,
    "races": races,
    "results": results,
    "constructors": constructors,
    "circuits": circuits,
}

print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

for name, df in dataframes.items():
    print("=" * 50)
    print(name.upper())
    print(f"Shape: {df.shape}")
    print("Data Types:")
    print(df.dtypes)
    print()


# Task2 - Data Cleaning

In [ ]:
results_cols = ["milliseconds", "fastestLapTime", "fastestLapSpeed"]
drivers_cols = ["dob", "nationality"]

def missing_report(df, cols):
    return pd.DataFrame({
        "missing_count": [(df[col].astype(str) == MISSING_VALUE).sum() for col in cols],
        "missing_pct": [((df[col].astype(str) == MISSING_VALUE).mean() * 100) for col in cols],
    }, index=cols).round(2)

print("=== results.csv ===")
print(f"Total rows: {len(results)}")
display(missing_report(results, results_cols))

print("=== drivers.csv ===")
print(f"Total rows: {len(drivers)}")
display(missing_report(drivers, drivers_cols))


### additional note - why results.csv has so many missing values?

#### 1. a lack of record from the past

In [ ]:
results_with_year = results.merge(
    races[["raceId", "year"]],
    on="raceId",
    how="left",
)

pre_2000 = results_with_year[results_with_year["year"] < 2000]
post_2000 = results_with_year[results_with_year["year"] >= 2000]

summary = []
for col in results_cols:
    summary.append({
        "column": col,
        "pre_2000_missing_pct": (pre_2000[col].astype(str) == MISSING_VALUE).mean() * 100,
        "post_2000_missing_pct": (post_2000[col].astype(str) == MISSING_VALUE).mean() * 100,
    })

missing_by_period = pd.DataFrame(summary).round(2)

print(f"Pre-2000 rows: {len(pre_2000)}")
print(f"Post-2000 rows: {len(post_2000)}")
missing_by_period


#### 2. a result of DNF

In [ ]:
dnf_summary = []

for col in results_cols:
    missing_mask = results[col].astype(str) == MISSING_VALUE
    n_missing = missing_mask.sum()
    n_dnf = (results.loc[missing_mask, "positionText"] == "R").sum()

    dnf_summary.append({
        "column": col,
        "total_missing": n_missing,
        "dnf_missing": n_dnf,
        "dnf_ratio_pct": (n_dnf / n_missing * 100) if n_missing else 0,
    })

pd.DataFrame(dnf_summary).round(2)


As you can see, the missing values in results.csv are highly influenced by lack of record before 2000. And high ratio of DNF caused a lot of missing values as well.


# Task3 - Merge Master Dataset

In [ ]:
drivers_named = drivers.assign(
    driverName=drivers["forename"] + " " + drivers["surname"]
)

merged_f1 = (
    results
    .merge(races[["raceId", "year", "round"]], on="raceId", how="left")
    .merge(drivers_named[["driverId", "driverName"]], on="driverId", how="left")
    .merge(
        constructors[["constructorId", "name"]].rename(
            columns={"name": "constructorName"}
        ),
        on="constructorId",
        how="left",
    )
)

merged_f1 = merged_f1[[
    "raceId",
    "year",
    "round",
    "driverName",
    "constructorName",
    "grid",
    "position",
    "points",
    "statusId",
]]

merged_f1.to_csv(OUTPUT_DIR / "merged_f1.csv", index=False)

print(merged_f1.shape)
merged_f1.head()


# Task4 - Top Drivers by Career Points

In [ ]:
result_races = results.merge(
    races[["raceId", "year"]],
    on="raceId",
    how="left",
)

result_races = result_races[
    result_races["year"].between(1995, 2020)
]

career_points = (
    result_races
    .groupby("driverId", as_index=False)
    .agg(Total_Points=("points", "sum"))
    .merge(drivers[["driverId", "forename", "surname"]], on="driverId", how="left")
)

career_points["driver"] = (
    career_points["forename"] + " " + career_points["surname"]
)

top10 = (
    career_points[["driver", "Total_Points"]]
    .sort_values("Total_Points", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

top10


In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=top10,
    x="Total_Points",
    y="driver",
    hue="driver",
    palette="viridis",
    legend=False,
)

plt.title("Top 10 Formula 1 Drivers by Career Points (1995-2020)")
plt.xlabel("Total Career Points")
plt.ylabel("Driver")
plt.tight_layout()
plt.show()


# Task5 - Top Constructors by Decade

In [ ]:
results_with_position = results.assign(
    position=pd.to_numeric(results["position"], errors="coerce")
)

wins = (
    results_with_position[results_with_position["position"] == 1]
    .merge(races[["raceId", "year"]], on="raceId", how="left")
    .merge(constructors[["constructorId", "name"]], on="constructorId", how="left")
)

wins["decade"] = (wins["year"] // 10 * 10).astype(int).astype(str) + "s"

wins_by_decade = (
    wins
    .groupby(["decade", "name"])
    .size()
    .reset_index(name="wins")
    .sort_values(["decade", "wins"], ascending=[True, False])
)

wins_pivot = (
    wins_by_decade
    .pivot(index="name", columns="decade", values="wins")
    .fillna(0)
    .astype(int)
)

wins_pivot


# Task6 - NumPy Statistics on Finishing Position

In [ ]:
results_with_finish = results.assign(
    positionOrder=pd.to_numeric(results["positionOrder"], errors="coerce")
)

driver_names = ["Lewis Hamilton", "Michael Schumacher"]
selected_drivers = drivers.assign(
    driver=drivers["forename"] + " " + drivers["surname"]
)
selected_drivers = selected_drivers[selected_drivers["driver"].isin(driver_names)]

def finish_positions(driver_name):
    driver_id = selected_drivers.loc[
        selected_drivers["driver"] == driver_name,
        "driverId",
    ].iloc[0]

    return results_with_finish.loc[
        results_with_finish["driverId"] == driver_id,
        "positionOrder",
    ].dropna().to_numpy()

comparison = pd.DataFrame([
    {
        "Driver": driver_name,
        "Mean Finish": np.mean(positions),
        "Median Finish": np.median(positions),
        "Std Dev": np.std(positions),
    }
    for driver_name in driver_names
    for positions in [finish_positions(driver_name)]
]).round(2)

comparison


# Task7 - Chart 1 - Races per Season

In [ ]:
races_per_season = (
    races
    .groupby("year", as_index=False)
    .agg(races=("raceId", "count"))
)

plt.figure(figsize=(12, 5))
sns.lineplot(data=races_per_season, x="year", y="races", marker="o")
plt.title("Formula 1 Races per Season")
plt.xlabel("Year")
plt.ylabel("Number of Races")
plt.tight_layout()
plt.show()
